# MALDI-TOF AMR Prediction with Conditional VAE + Prototypes

This notebook implements a structured deep learning model for predicting antimicrobial resistance (AMR) from MALDI-TOF spectra.

## Key ideas:

- A **Variational Autoencoder (VAE)** learns a latent representation `z` of spectra.
- A second latent space `w` is conditioned on both:
  - the spectrum (`z`)
  - the antibiotic embedding
- A classifier predicts resistance from `[z, w, antibiotic_embedding]`
- A **prototype structure** is imposed in `w`:
  - each antibiotic has:
    - one prototype for Sensitive (S)
    - one for Resistant (R)


## Imports

In [1]:
import os
import pickle
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

## Reproducibility 

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cpu


## Config

In [3]:
DATA_PATH = "/export/usuarios01/egarroyo/MALDI_for_AMR_prediction/data/COMBINED_MARISMA_DRIAMS.pkl"

TARGET_SPECIES = "Klebsiella_Pneumoniae"

TARGET_ANTIBIOTICS = [
    "Ciprofloxacin",
    "Ceftriaxone",
    "Piperacillin-Tazobactam",
    "Cefepime",
    "Imipenem",
    "Meropenem"
]

LATENT_DIM_Z = 64
LATENT_DIM_W = 64
AB_EMB_DIM = 64

BATCH_SIZE = 64
EPOCHS = 50
LR = 1e-3

BETA_Z = 0.01
BETA_W = 0.01

LAMBDA_REC = 1.0
LAMBDA_AMR = 1.0

LAMBDA_PROTO_PULL = 1.0
LAMBDA_PROTO_PUSH = 1.0
PROTO_MARGIN = 2.0

TEST_SIZE = 0.2

## Data loading

In [4]:
with open(DATA_PATH, "rb") as f:
    payload = pickle.load(f)

X = payload["data"]
y_species = payload["label"]
amr = payload["amr"]
antibiotics = list(payload["antibiotics"])

print("Shape X:", X.shape)
print("Num antibiotics:", len(antibiotics))

Shape X: (66832, 6000)
Num antibiotics: 104


## Preprocessing

In [5]:
def remove_top_k_peaks_per_spectrum(X, top_k=1):
    X_new = X.copy()
    for i in range(X_new.shape[0]):
        idx = np.argpartition(X_new[i], -top_k)[-top_k:]
        X_new[i, idx] = 0
    return X_new

X = remove_top_k_peaks_per_spectrum(X, top_k=1)

# Filter species
mask = np.array(y_species) == TARGET_SPECIES
X = X[mask]
amr = amr[mask]

# Select antibiotics
ab_idx = [antibiotics.index(a) for a in TARGET_ANTIBIOTICS]
amr = amr[:, ab_idx]

num_antibiotics = len(TARGET_ANTIBIOTICS)

# Transform
#X = np.log1p(X).astype(np.float32)
#amr = amr.astype(np.float32)

print("Filtered shape:", X.shape)

Filtered shape: (19265, 6000)


## Split

In [6]:
# Stratify by "has resistance somewhere"
strat = np.nan_to_num(amr, nan=0).sum(axis=1) > 0

train_idx, test_idx = train_test_split(
    np.arange(len(X)),
    test_size=TEST_SIZE,
    stratify=strat
)

X_train, X_test = X[train_idx], X[test_idx]
amr_train, amr_test = amr[train_idx], amr[test_idx]

## Dataset

In [7]:
class DS(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X)
        self.y = torch.tensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y[i]

train_loader = DataLoader(DS(X_train, amr_train), batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(DS(X_test, amr_test), batch_size=BATCH_SIZE)

## Model Architecture

We implement a **Conditional VAE with prototype structure**:

- `z`: global latent representation of spectra
- `w`: antibiotic-conditioned latent space
- classifier uses `[z, w, embedding]`

In [8]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()

        # Encoder z
        self.enc = nn.Sequential(
            nn.Linear(X.shape[1],1024), nn.ReLU(),
            nn.Linear(1024,256), nn.ReLU()
        )

        self.z_mu = nn.Linear(256, LATENT_DIM_Z)
        self.z_logvar = nn.Linear(256, LATENT_DIM_Z)

        # Decoder
        self.dec = nn.Sequential(
            nn.Linear(LATENT_DIM_Z,256), nn.ReLU(),
            nn.Linear(256,1024), nn.ReLU()
        )

        self.mu = nn.Linear(1024, X.shape[1])
        self.sig = nn.Linear(1024, X.shape[1])

        # Antibiotic embedding
        self.ab_emb = nn.Embedding(num_antibiotics, AB_EMB_DIM)

        # Encoder w
        self.enc_w = nn.Sequential(
            nn.Linear(LATENT_DIM_Z+AB_EMB_DIM,128),
            nn.ReLU(),
            nn.Linear(128,64)
        )

        self.w_mu = nn.Linear(64, LATENT_DIM_W)
        self.w_logvar = nn.Linear(64, LATENT_DIM_W)

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(LATENT_DIM_Z+LATENT_DIM_W+AB_EMB_DIM,64),
            nn.ReLU(),
            nn.Linear(64,1)
        )

        # Prototypes
        self.prototypes = nn.Parameter(
            torch.randn(num_antibiotics,2,LATENT_DIM_W)*0.01
        )

    def reparam(self, mu, lv):
        return mu + torch.randn_like(mu) * torch.exp(0.5*lv)

    def encode_z(self, x):
        h = self.enc(x)
        mu = self.z_mu(h)
        lv = self.z_logvar(h)
        z = self.reparam(mu, lv)
        return z, mu, lv

    def decode(self, z):
        h = self.dec(z)
        mu = self.mu(h)
        s = F.softplus(self.sig(h))
        return mu, s

    def encode_w(self, z, a):
        e = self.ab_emb(a)
        h = self.enc_w(torch.cat([z,e],1))
        mu = self.w_mu(h)
        lv = self.w_logvar(h)
        w = self.reparam(mu, lv)
        return w, mu, lv, e

    def pred(self, z, w, e):
        return torch.sigmoid(self.classifier(torch.cat([z,w,e],1)))

## Losses 

In [9]:
def KL(mu, lv):
    return -0.5 * torch.mean(1 + lv - mu**2 - lv.exp())

def recon_loss(x, mu, s):
    logx = torch.log(x + 1e-8)
    return ((logx - mu)**2 / (2*s**2) + torch.log(s)).mean()

def BCE(p, t):
    m = ~torch.isnan(t)
    return F.binary_cross_entropy(p[m], t[m])

def proto_loss(w, a, t, proto):
    m = ~torch.isnan(t.squeeze())
    if m.sum() == 0:
        return 0,0

    w,a,t = w[m],a[m],t[m].long().squeeze()
    p = proto[a,t]

    pull = ((w-p)**2).sum(1).mean()

    s = proto[:,0]
    r = proto[:,1]

    push = F.relu(PROTO_MARGIN - torch.norm(s-r,dim=1)).mean()

    return pull, push

## Training

In [10]:
model = Model().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)

for ep in range(EPOCHS):
    model.train()
    total_loss = 0

    for x,y in train_loader:

        x = x.to(DEVICE)
        y = y.to(DEVICE)

        opt.zero_grad()

        # --- Z ---
        z, zm, zl = model.encode_z(x)

        # --- Reconstruction ---
        mu, s = model.decode(z)
        loss_rec = recon_loss(x, mu, s)

        # --- Expand for antibiotics ---
        B = z.size(0)

        z_exp = z.unsqueeze(1).repeat(1,num_antibiotics,1).reshape(-1,LATENT_DIM_Z)
        a = torch.arange(num_antibiotics).to(DEVICE).repeat(B)

        y_flat = y.reshape(-1,1)

        # --- W ---
        w, wm, wl, e = model.encode_w(z_exp, a)

        # --- Prediction ---
        p = model.pred(z_exp, w, e)

        loss_amr = BCE(p, y_flat)

        # --- KL ---
        loss_kl = BETA_Z*KL(zm,zl) + BETA_W*KL(wm,wl)

        # --- Prototypes ---
        lpull, lpush = proto_loss(w, a, y_flat, model.prototypes)

        loss = (
            LAMBDA_REC*loss_rec +
            loss_kl +
            LAMBDA_AMR*loss_amr +
            LAMBDA_PROTO_PULL*lpull +
            LAMBDA_PROTO_PUSH*lpush
        )

        loss.backward()
        opt.step()

        total_loss += loss.item()

    print(f"Epoch {ep}: {total_loss:.3f}")

Epoch 0: 3063.534


KeyboardInterrupt: 

## Eval

In [ ]:
model.eval()

preds = [[] for _ in range(num_antibiotics)]
true  = [[] for _ in range(num_antibiotics)]

with torch.no_grad():
    for x,y in test_loader:

        x = x.to(DEVICE)
        z,_,_ = model.encode_z(x)

        for i in range(num_antibiotics):

            a = torch.full((z.size(0),), i).to(DEVICE)
            w,_,_,e = model.encode_w(z, a)

            p = model.pred(z, w, e).cpu().numpy()

            preds[i] += list(p)
            true[i]  += list(y[:,i].numpy())

print("\nAUC:")
for i,a in enumerate(TARGET_ANTIBIOTICS):
    yt = np.array(true[i])
    yp = np.array(preds[i])

    m = ~np.isnan(yt)

    if len(np.unique(yt[m])) > 1:
        print(a, roc_auc_score(yt[m], yp[m]))

## Prototype distances 

In [ ]:
proto = model.prototypes.detach().cpu().numpy()

print("\nPrototype distances:")
for i, ab in enumerate(TARGET_ANTIBIOTICS):
    d = np.linalg.norm(proto[i,0] - proto[i,1])
    print(ab, round(float(d),4))

## Latent Space Visualization (z)

We project the global latent space `z` into 2D using UMAP or t-SNE.

This shows how spectra cluster independently of antibiotics.

In [ ]:
from sklearn.manifold import TSNE

try:
    import umap.umap_ as umap
    reducer = umap.UMAP(n_components=2, random_state=42)
    REDUCER_NAME = "UMAP"
except:
    reducer = TSNE(n_components=2, random_state=42)
    REDUCER_NAME = "t-SNE"

# Collect z
Z_all = []
Y_all = []

model.eval()
with torch.no_grad():
    for x,y in test_loader:
        x = x.to(DEVICE)
        z,_,_ = model.encode_z(x)

        Z_all.append(z.cpu().numpy())
        Y_all.append(y.numpy())

Z_all = np.vstack(Z_all)
Y_all = np.vstack(Y_all)

Z_2D = reducer.fit_transform(Z_all)

plt.figure(figsize=(7,6))
plt.scatter(Z_2D[:,0], Z_2D[:,1], alpha=0.5)
plt.title(f"Latent space z ({REDUCER_NAME})")
plt.xlabel("Dim 1")
plt.ylabel("Dim 2")
plt.show()

## Latent Space w (antibiotic-conditioned)

We visualize the latent space `w` for a specific antibiotic.

Colors represent:
- Blue → Sensitive
- Red → Resistant

In [ ]:
AB_TO_VISUALIZE = 0  # change index to explore

W_all = []
Y_ab = []

with torch.no_grad():
    for x,y in test_loader:
        x = x.to(DEVICE)
        z,_,_ = model.encode_z(x)

        a = torch.full((z.size(0),), AB_TO_VISUALIZE).to(DEVICE)
        w,_,_,_ = model.encode_w(z, a)

        W_all.append(w.cpu().numpy())
        Y_ab.append(y[:,AB_TO_VISUALIZE].numpy())

W_all = np.vstack(W_all)
Y_ab = np.hstack(Y_ab)

W_2D = reducer.fit_transform(W_all)

mask = ~np.isnan(Y_ab)

plt.figure(figsize=(7,6))

plt.scatter(
    W_2D[mask & (Y_ab==0),0],
    W_2D[mask & (Y_ab==0),1],
    alpha=0.6,
    label="Sensitive"
)

plt.scatter(
    W_2D[mask & (Y_ab==1),0],
    W_2D[mask & (Y_ab==1),1],
    alpha=0.6,
    label="Resistant"
)

plt.title(f"w space for {TARGET_ANTIBIOTICS[AB_TO_VISUALIZE]}")
plt.legend()
plt.show()

## Prototypes vs Samples

We project prototypes and samples together in the same space.

This allows us to see:
- how well samples cluster around prototypes
- separation between S and R

In [ ]:
proto = model.prototypes.detach().cpu().numpy()

# take same antibiotic
proto_ab = proto[AB_TO_VISUALIZE]

# stack samples + prototypes
combined = np.vstack([W_all, proto_ab])

combined_2D = reducer.fit_transform(combined)

W_2D = combined_2D[:-2]
P_2D = combined_2D[-2:]

plt.figure(figsize=(7,6))

plt.scatter(
    W_2D[:,0],
    W_2D[:,1],
    alpha=0.3,
    label="Samples"
)

plt.scatter(
    P_2D[0,0],
    P_2D[0,1],
    s=200,
    marker="X",
    label="Prototype S"
)

plt.scatter(
    P_2D[1,0],
    P_2D[1,1],
    s=200,
    marker="X",
    label="Prototype R"
)

plt.title(f"Prototypes in w space ({TARGET_ANTIBIOTICS[AB_TO_VISUALIZE]})")
plt.legend()
plt.show()

## Prototype Distance Matrix

We compute distances between S and R prototypes for all antibiotics.

This measures how separable each antibiotic is.

In [ ]:
import seaborn as sns

proto = model.prototypes.detach().cpu().numpy()

distances = []

for i in range(num_antibiotics):
    d = np.linalg.norm(proto[i,0] - proto[i,1])
    distances.append(d)

plt.figure(figsize=(8,4))
sns.barplot(x=TARGET_ANTIBIOTICS, y=distances)
plt.xticks(rotation=45)
plt.title("Prototype distances (S vs R)")
plt.ylabel("Distance")
plt.show()

## Distance of Samples to Prototypes

We compute how close each sample is to its corresponding prototype.

This gives insight into:
- confidence
- ambiguity regions

In [ ]:
dist_S = []
dist_R = []

proto = model.prototypes.detach().cpu()

with torch.no_grad():
    for x,y in test_loader:
        x = x.to(DEVICE)
        z,_,_ = model.encode_z(x)

        a = torch.full((z.size(0),), AB_TO_VISUALIZE).to(DEVICE)
        w,_,_,_ = model.encode_w(z,a)

        y_ab = y[:,AB_TO_VISUALIZE]

        for i in range(len(w)):
            if np.isnan(y_ab[i]):
                continue

            label = int(y_ab[i])
            p = proto[AB_TO_VISUALIZE, label]

            d = torch.norm(w[i].cpu() - p).item()

            if label == 0:
                dist_S.append(d)
            else:
                dist_R.append(d)

plt.figure(figsize=(7,5))
plt.hist(dist_S, bins=30, alpha=0.6, label="Sensitive")
plt.hist(dist_R, bins=30, alpha=0.6, label="Resistant")
plt.legend()
plt.title("Distance to correct prototype")
plt.show()